In [121]:
import pandas as pd
from pathlib import Path
import json
import pyarrow as pa
from frozendict import frozendict

In [122]:
paths = Path("./testOutputs/child_benefit/").glob("*__Model=*Commit*/Permutation*.conversation.json")


In [123]:
paths_list = list(paths)

In [124]:
len(paths_list)

5776

In [125]:
foo = json.load(paths_list[0].open())

print(json.dumps(foo, indent=2))

{
  "case_id": "MULTI_MIXED_THREE_MIXED",
  "meta": {
    "permutation": 46,
    "test_case": {
      "case_id": "MULTI_MIXED_THREE_MIXED",
      "facts": {
        "claimant_lives_in_uk": {
          "description": "Whether the claimant lives in the UK.",
          "value": true
        },
        "children": {
          "description": "A list of children the claimant is claiming for.",
          "value": [
            {
              "id": {
                "description": "An identifier for the child.",
                "value": "child_0"
              },
              "name": {
                "description": "The child's first name.",
                "value": "Alex"
              },
              "age": {
                "description": "The child's age in years.",
                "value": 5
              },
              "lives_with_claimant": {
                "description": "Whether the child lives with the claimant.",
                "value": true
              },
              "i

In [126]:
dfs = [
    pd.json_normalize(
        json.load(
            p.open()
        )
    )
    for p in paths_list
]
df = pd.concat(dfs)

In [127]:
pd.set_option('display.max_colwidth', None)

In [128]:
df["meta.conversation.test_case.expected_eligibility"]

0    NaN
0    NaN
0    NaN
0    NaN
0    NaN
    ... 
0    NaN
0    NaN
0    NaN
0    NaN
0    NaN
Name: meta.conversation.test_case.expected_eligibility, Length: 5776, dtype: object

In [129]:
df_with_eligibility = df.dropna(subset=["meta.conversation.test_case.expected_eligibility"])

In [130]:
df_with_eligibility["eligibility_agent_payload.response.response.child_evaluations"] = df_with_eligibility["eligibility_agent_payload.response.response.child_evaluations"].astype(
    pd.ArrowDtype(pa.list_(
        pa.struct(
            {
                "child_id": pa.string(), 
                "name": pa.string(), 
                "eligibility": pa.bool_()
            }
        )
    )
))

/var/folders/b2/7k_brtrs25x6v118yqs3yt8m0000gp/T/ipykernel_87130/852077009.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_eligibility["eligibility_agent_payload.response.response.child_evaluations"] = df_with_eligibility["eligibility_agent_payload.response.response.child_evaluations"].astype(


In [131]:
df_with_eligibility["meta.conversation.test_case.expected_eligibility"] = df_with_eligibility["meta.conversation.test_case.expected_eligibility"].astype(
    pd.ArrowDtype(pa.list_(
        pa.struct(
            {
                "child_id": pa.string(), 
                "name": pa.string(), 
                "eligible": pa.bool_()
            }
        )
    )
))

/var/folders/b2/7k_brtrs25x6v118yqs3yt8m0000gp/T/ipykernel_87130/269028256.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_eligibility["meta.conversation.test_case.expected_eligibility"] = df_with_eligibility["meta.conversation.test_case.expected_eligibility"].astype(


In [132]:
df_with_eligibility["human.expected_eligibility"] = df_with_eligibility["meta.conversation.test_case.expected_eligibility"].apply(
    lambda children: set(
        [
            frozendict(
                {"eligible": child["eligible"], "child_id": child["child_id"]}
            ) 
            for child in children
        ]
    )
)

/var/folders/b2/7k_brtrs25x6v118yqs3yt8m0000gp/T/ipykernel_87130/732599712.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_with_eligibility["human.expected_eligibility"] = df_with_eligibility["meta.conversation.test_case.expected_eligibility"].apply(


In [136]:
df_with_eligibility["human.actual_eligibility"] = df_with_eligibility["eligibility_agent_payload.response.response.child_evaluations"].apply(
    lambda children: set(
        [
            frozendict(
                {"eligible": child["eligibility"], "child_id": child["child_id"]}
            )
            for child in children
        ]
    )
)

TypeError: 'NAType' object is not iterable

In [ ]:
#
lhs = df_with_eligibility["human.expected_eligibility"] #df["meta.conversation.test_case.expected_eligibility"].apply(lambda v: {"eligible": v["is_eligible"], "child_id": v["child_id"]})
rhs = df_with_eligibility["human.actual_eligibility"] #df["eligibility_agent_payload.response.response.child_evaluations"].apply(lambda v: {"eligible": v["eligible"], "child_id": v["child_id"]})

lhs[["child_id", "is_eligible"]] != rhs[["child_id", "eligible"]]
#lhs.dropna()

In [ ]:
df[df["meta.conversation.test_case.expected_eligibility"] != df["meta.conversation.test_case.expected_eligibility"]]